# Phase 2 — SAP Hybrid RAG (Evidence Retrieval)

Read SAP from the existing private Drive folder. Compare BM25, dense and RRF hybrid retrieval using identical chunks and query facets. This is candidate evidence only, not Step 2 extraction or validated clinical requirements. No API key needed; pretrained weights are downloaded for local inference. Audit outputs contain SAP excerpts; do not share or commit without authorization.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib
REPO_URL = 'https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH = 'generic-retrieval-v2'
ROOT = Path('/content/tfl-generic-retrieval-v2')
if not ROOT.exists():
    subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(ROOT)])
else:
    current = subprocess.check_output(['git','-C',str(ROOT),'branch','--show-current'], text=True).strip()
    assert current == BRANCH, f'Wrong branch: {current}'
    subprocess.check_call(['git','-C',str(ROOT),'pull','--ff-only'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[notebook,test]','sentence-transformers>=3,<6'])
SRC = (ROOT / 'src').resolve()
assert (SRC / 'tfl_agent/tools/sap_hybrid.py').is_file(), SRC
if str(SRC) in sys.path:
    sys.path.remove(str(SRC))
sys.path.insert(0, str(SRC))
importlib.invalidate_caches()
cached = sys.modules.get('tfl_agent')
if cached is not None and not Path(cached.__file__).resolve().is_relative_to(SRC):
    raise RuntimeError('Wrong tfl_agent checkout already imported. Restart session and run all.')
import tfl_agent
from tfl_agent.agent.sap_retrieval_pilot import run_sap_retrieval
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC)
print('Commit:', subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
print('Agent package loaded:', tfl_agent.__file__)

## 1. Configure the existing Drive SAP

For SAP_001_00.pdf use physical pages 4–21, excluding the Content List and shells. For a different study, edit the range and targets together. These targets are the previous demo targets; the failed Phase 1 in-library cases are not assumed to belong to this SAP.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_SAP_RELATIVE_PATH = 'Colab Notebooks/agent/midterm demo/SAP_001_00.pdf'
SAP_PATH = Path('/content/drive/MyDrive') / DRIVE_SAP_RELATIVE_PATH
SAP_BODY_PAGE_RANGE = (4, 21)
TARGETS = [
    {'tfl_id':'14.1.3.1','title':'Demographic and Baseline Characteristics – Safety Analysis Set'},
    {'tfl_id':'14.2.3.1','title':'Summary of Progression Free Survival – ITT Analysis Set'},
]
if not SAP_PATH.is_file():
    available = sorted(p.name for p in SAP_PATH.parent.glob('*.pdf'))
    raise FileNotFoundError(f'{SAP_PATH}; PDFs in folder: {available}')
SAVE_TO_DRIVE = True
print('SAP:', SAP_PATH)
print('Physical pages:', SAP_BODY_PAGE_RANGE)

## 2. Run retrieval

The model is loaded once and the SAP index is reused for both targets. If the model download fails, the notebook stops rather than substituting BM25 silently.

In [ ]:
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = Path('/content/SAP_Hybrid_Phase2') / RUN_ID
summary, rows = run_sap_retrieval(SAP_PATH, SAP_BODY_PAGE_RANGE, TARGETS, OUT)
print(json.dumps(summary, indent=2))
df = pd.DataFrame(rows)
for t in TARGETS:
    display(df[(df.tfl_id == t['tfl_id']) & (df['rank'] == 1)][
        ['facet','method','rank','pages','section','chunk_id','text']])

## 3. Compare methods and validate provenance

Top-1 agreement is descriptive, not relevance accuracy. Use the exported Evidence_Annotation.csv to label retrieved chunks if authorized. No independent ground-truth labels are supplied by this notebook.

In [ ]:
pivot = df[df['rank'] == 1].pivot(index=['tfl_id','facet'],columns='method',values='chunk_id')
display(pivot)
chunks = json.loads((OUT/'SAP_Chunks.json').read_text())
blocks = {b['block_id']:b for b in json.loads((OUT/'SAP_Blocks.json').read_text())}
for c in chunks:
    reconstructed = '\n'.join(blocks[s['block_id']]['text'][s['start']:s['end']] for s in c['spans'])
    assert reconstructed == c['text'], c['chunk_id']
    assert all(SAP_BODY_PAGE_RANGE[0] <= p <= SAP_BODY_PAGE_RANGE[1] for p in c['pages'])
print('All chunk spans and physical pages verified.')
subprocess.check_call([sys.executable,'-m','pytest','-q','tests/test_sap_hybrid_phase2.py','tests/test_generic_library_phase1.py'],cwd=ROOT)

## 4. Export results

The ZIP is saved in the same private Drive directory by default and downloaded to your computer. It includes SAP text; only share files you are authorized to disclose.

In [ ]:
import shutil
from google.colab import files
ZIP_PATH = Path(shutil.make_archive(str(OUT), 'zip', OUT))
print('Local output:', OUT)
print('ZIP:', ZIP_PATH)
if SAVE_TO_DRIVE:
    destination = SAP_PATH.parent / 'Phase2_Outputs'
    destination.mkdir(parents=True, exist_ok=True)
    saved = Path(shutil.copy2(ZIP_PATH, destination / ZIP_PATH.name))
    print('Saved to Google Drive:', saved)
files.download(str(ZIP_PATH))